# Full benchmarking timeline (entry-point notebook)

This notebook walks you through the entire kermit benchmarking pipeline end-to-end:

1. **Generate** a LUBM-1 dataset on demand (`bench gen lubm`).
2. **Run** the time and space benchmarks across both index structures.
3. **Load** the resulting reports into a pandas DataFrame.
4. **Plot** four inline `matplotlib` figures.

Run the cells top-to-bottom (`Run All`) to reproduce. The first invocation triggers a full release build (~3-5 min); subsequent runs short-circuit on the spec-hash cache.

## Prerequisites

1. **Inside `nix develop`** if on NixOS — sets `LD_LIBRARY_PATH` for `libstdc++` and `libz` so numpy/matplotlib wheels load.
2. **JDK 8 on PATH** — the flake provides `pkgs.jdk8`; on Ubuntu, `apt install openjdk-8-jre`.
3. **Kermit-lab venv populated**: from the repo root, `cd python/kermit-lab && uv sync` (one-time).
4. **Launch this notebook** with: `cd python/kermit-lab && uv run --with jupyter jupyter lab notebooks/`, then open `00_full_timeline.ipynb`.

This notebook is the entry point. The other five notebooks under `notebooks/` (`01_quick_start`, `02_scaling`, ...) are topical deep-dives that assume `bench-runs/` and `target/criterion/` are already populated by an earlier run.

In [ ]:
import os, subprocess, shutil, time
from pathlib import Path

# Resolve the repo root by walking up until we find Cargo.toml.
# This makes the notebook robust to wherever the kernel was launched from.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "Cargo.toml").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("Could not locate Kermit repo root (no Cargo.toml found above CWD)")
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
print(f"CWD: {os.getcwd()}")

# Fail loud upfront if the toolchain is missing — better than a confusing
# error 30 seconds into phase 1.
assert shutil.which("cargo"), "cargo not on PATH — run from inside `nix develop`"
assert shutil.which("java"),  "java not on PATH — needed by `bench gen lubm`"
print("cargo:", shutil.which("cargo"))
print("java: ", shutil.which("java"))


def run_phase(args, label):
    """Run a kermit subprocess with timing and streamed output.

    Used by phases 1, 2, and 3 — each invokes `cargo run --release -- bench …`
    and may take several minutes on a cold build. Output is streamed live to
    the notebook so the cell does not appear hung. We keep a rolling buffer
    of the last 40 lines and surface it on non-zero exit before raising
    CalledProcessError.
    """
    print(f"⏳ {label} - this may take several minutes...")
    start = time.monotonic()
    proc = subprocess.Popen(
        args,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    last_lines: list[str] = []
    for line in proc.stdout:
        line = line.rstrip()
        last_lines.append(line)
        if len(last_lines) > 40:
            last_lines.pop(0)
        print(line)
    rc = proc.wait()
    elapsed = time.monotonic() - start
    if rc != 0:
        raise subprocess.CalledProcessError(rc, args, output="\n".join(last_lines))
    print(f"✅ {label} - done in {elapsed:.1f}s")
    return proc
